# Regularization & cross-validation

_Machine Learning_

**Penalize big weights to fight overfitting. Use folds to tune the penalty.**

Overfitting often shows up as huge, wild weights.

     Regularization adds a penalty for big weights to the cost.

---

This notebook is a practice scaffold for the **Regularization & cross-validation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Build a problem that overfits

Overfitting is worst when you have **few examples but many features**. We make exactly that: 60 samples with 40 features, then hold out 40% for testing. With so little data per feature, plain least squares can fit the training noise and generalise poorly.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Few samples, many features -> plain least squares overfits.
X, y = make_regression(n_samples=60, n_features=40, noise=10.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

print("train rows:", Xtr.shape[0], " features:", Xtr.shape[1])

### Step 2 — Fit plain least squares and ridge side by side

`LinearRegression` is ordinary least squares (OLS), no penalty. `RidgeCV` adds an L2 penalty on the weights and uses cross-validation to pick the penalty strength `alpha` from a grid. The cross-validation means ridge tunes its own regularization without ever touching the test set.

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV

# OLS: no penalty on the weights.
ols = LinearRegression().fit(Xtr, ytr)

# Ridge: L2 penalty, with alpha chosen by cross-validation over a grid.
ridge = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(Xtr, ytr)

### Step 3 — Compare test R² and weight sizes

We score both models on the held-out test set and also measure the **norm** (overall size) of their weight vectors. The lesson: ridge usually gets a higher test R², and its weight norm is much smaller — the penalty literally shrinks the weights, which is what fights overfitting.

In [ ]:
from sklearn.metrics import r2_score

# Test R^2 for each model (higher = generalises better).
ols_r2 = r2_score(yte, ols.predict(Xte))
ridge_r2 = r2_score(yte, ridge.predict(Xte))
print("OLS    test R^2:", round(ols_r2, 3))
print("Ridge  test R^2:", round(ridge_r2, 3))

# The penalty strength ridge picked by cross-validation.
print("chosen penalty alpha:", round(ridge.alpha_, 3))

# Weight norm: ridge's penalty shrinks the weights, so its norm is smaller.
ols_norm = float(np.linalg.norm(ols.coef_))
ridge_norm = float(np.linalg.norm(ridge.coef_))
print("OLS    weight norm:", round(ols_norm, 2))
print("Ridge  weight norm:", round(ridge_norm, 2))

## Visualize the data & results

_On diabetes data, does shrinking the weights (ridge) beat plain least squares, and how much penalty is too much?_

### Step 1 — Prepare the real diabetes data and fit both models

We load the 442-patient diabetes data and **standardise** the features (zero mean, unit variance) so the L2 penalty treats them on equal footing. Then we fit OLS and a cross-validated ridge on the same train / test split, ready to compare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

db = load_diabetes()
# Standardise so the L2 penalty treats every feature on equal footing.
X = StandardScaler().fit_transform(db.data)
y = db.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

ols = LinearRegression().fit(Xtr, ytr)
ridge = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(Xtr, ytr)

### Step 2 — Bar chart: OLS vs cross-validated ridge

A direct head-to-head of the two models' test R². On this real data the gap is smaller than on the deliberately-overfit synthetic problem, but it shows ridge holding its own while keeping the weights tame.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: head-to-head test R^2 for OLS vs the cross-validated ridge.
ols_r2 = r2_score(yte, ols.predict(Xte))
ridge_r2 = r2_score(yte, ridge.predict(Xte))
ax1.bar(["OLS", "ridge (CV)"],
        [ols_r2, ridge_r2],
        color=["#ff7b72", "#7ee787"])
ax1.set_ylabel("test R squared")
ax1.set_title("OLS vs ridge")

### Step 3 — Sweep the penalty to find the sweet spot

We fit a fresh ridge at each `alpha` across a log-spaced range and plot test R² against it. The curve rises as a little regularization helps, then falls once `alpha` grows so large it crushes the weights toward zero (underfitting). The peak is the sweet spot cross-validation aims for.

In [ ]:
# Sweep alpha across a wide log-spaced range, refitting ridge each time.
alphas = np.logspace(-2, 3, 12)
r2s = [r2_score(yte, Ridge(alpha=a).fit(Xtr, ytr).predict(Xte)) for a in alphas]

# Right: test R^2 vs penalty strength -- too much penalty underfits.
ax2.plot(alphas, r2s, color="#c89bff", marker="o", label="test R squared")
ax2.set_xscale("log")
ax2.set_xlabel("alpha (penalty strength)")
ax2.set_ylabel("test R squared")
ax2.set_title("R squared vs ridge penalty")
ax2.legend()
plt.show()